# COMS W2132 Intermediate Computing in Python
## Priority Queues and Heaps
**Date**: March 30 2026
**Instructor**: Daniel Bauer

**Reading**: Data Structures and Algorithms in Python, 9.1, 9.3, (possibly 9.2.2-9.2.3)

---

## Recall Stacks and Queues 

Stacks and queues are sequence data types that allow interaction only at the end of the sequence. Operations should be implemented in O(1).

* **Stack**: Last-In-First-Out (LIFO)
    * `push(x)` add x on top of the stack  (called `append(x)` when using Python list)
    * `pop()` return top most element remove it from the stack.
    * `top()` return top most element, but don't remove it (using indexing `stack[-1]` when using Python list)
    * `len()` return number of elements on the stack.

  Stacks can be implemented using a linked list or an array list.
  
  Applications: processing nested structures, recursion, depth-first tree traversals,...

* **Queue**: First-In-First-Out (FIFO)
    * `enqueue(x)` add x at the end of the queue.
    * `dequeue()` return and remove the element at the front of the queue. 
    * `front()/peek()` return the element at the front of the queue but don't return it. 
    * `len()` return number of elements on the queue.
  
  Queues can be implemented using a doubly linked list or a circular array (or using two stacks).

  Applications: keeping track of to-do lists, simulations, breadth-first (layer-order) tree traversals, ...

* **Double Ended Queue / Deque**: combines stack and queue operations. Typically implemented using a doubly linked list. For example [`collections.deque`](https://docs.python.org/3/library/collections.html#collections.deque).

## Priority Queues

Like a stack or queue, the Priority Queue abstract data type has a pair of methods for inserting and retrieving elements, called `insert` and `remove_min`.
                                                             
In a priority queue, each element has an associated *priority* when inserted into the data structure. We can represent an element with its priority as a *(key,value)* pair, where the key is the priority (but note that, unlike maps, priority queues can generally contain multiple values with the same key).

**Example:** Gawain, Percivale, and Galahad arrive at a lunch buffet, waiting in line. They are all equally important (priority 10). 

```
pq = SomePriorityQueue()
pq.insert((10,"Gawain"))
pq.insert((10,"Percivale"))
pq.insert((10,"Galahad"))
```
They are served in first-come-first-serve order before they sit down at the round table. 

```
(k,v) = pq.remove_min()  # removes and returns (10, Gawain)
(k,v) = pq.remove_min()  # removes and returns (10, Percivale)
```

All of a sudden, King Arthur walks in and cuts the line (because he is King, he is more important)

```
pq.insert((1,"Arthur"))
```

Before Arthur is served, Lancelot arrives (arguably the most important one of the knights, after the king).

```
pq.insert((5,"Lancelot"))
```

Now Arthur is served, followed by Lancelot, and 

```
(k,v) = pq.remove_min()  # removes and returns (1,Arthur)
(k,v) = pq.remove_min()  # removes and returns (5,Lancelot)
(k,v) = pq.remove_min()  # removes and returns (10,Galahad)
```

The priority queue may also have a `find_min` method to retrieve but not remove the smallest element, and a `len` method to retrieve the number of elements in the priority queue.

There are many other use cases for priority queues, for example process managment on a CPU or prioritizing network packages by importance. 

Another important application is **sorting**, as we will see later. 

### Implementing a Priority Queue using an Unsorted List

Store the items in a list, simply append on insert, and search for minimum during lookup. 

In [7]:
class UnsortedPriorityQueue:

    class _Item:
        def __init__(self, priority,v): 
            self._priority = priority
            self._value = v
            
        def __lt__(self, other):            
            return self._priority < other._priority
    
    def __init__(self):
        self._data = [] 

    def __len__(self): 
        return len(self._data)
    
    def insert(self, priority,v): # k is the priority 
        self._data.append(self._Item(priority,v))

    def find_min(self): 
        current_smallest_index = 0 
        current_smallest_item = self._data[0]

        index = 1
        while index < len(self._data): 
            if self._data[index] < current_smallest_item:
                current_smallest_index = index 
                current_smallest_item = self._data[index]
            index += 1
            
        return current_smallest_item 

    def remove_min(self):
        current_smallest_index = 0 
        current_smallest_item = self._data[0]

        index = 1
        while index < len(self._data): 
            if self._data[index] < current_smallest_item:
                current_smallest_index = index 
                current_smallest_item = self._data[index]
            index += 1
            
        del self._data[current_smallest_index]
        return current_smallest_item
        
        

In [8]:
item1 = UnsortedPriorityQueue._Item(10,"Gallahad")
item2 = UnsortedPriorityQueue._Item(5, "Lancelot")

item2 < item1

True

In [14]:
pq = UnsortedPriorityQueue()
pq.insert(10, "Gallahad")
pq.insert(1, "Arthur")
pq.insert(5, "Lancelot")

In [15]:
item = pq.remove_min()
print(item._priority, item._value)

1 Arthur


In [16]:
item = pq.remove_min()
print(item._priority, item._value)

5 Lancelot


In [17]:
item = pq.remove_min()
print(item._priority, item._value)

10 Gallahad


Running times:

* `insert`: O(1) amortized 
* `find_min`: O(n)
* `remove_min`: O(n)


### Implementing a Priority Queue using a Sorted List

What if we maintain a sorted list, on insert we find the correct insertion site. On lookup the smallest item will be at the beginning. 

In [301]:
class SortedPriorityQueue:

    class _Item:
        def __init__(self, k,v): 
            self._key = k
            self.value = v
            
        def __lt__(self, other):            
            return self._key < other._key
    
    def __init__(self):
        self._data = [] 

    def __len__(self): 
        return len(self._data)
    
    def insert(self, k,v):   # O(N)

        new_item = self._Item(k,v) 
        
        current_index = 0
        while current_index < len(self._data) and self._data[current_index] < new_item: 
            current_index += 1        
        self._data.insert(current_index, new_item)
        
    def find_min(self):    # O(1)
        return self._data[0]

    def remove_min(self):  # O(N) 
        min = self._data[0]
        self._data.pop(0) # remove element at index 0
        return min
        
        

Running times:

* `insert`: O(n)
* `find_min`:  O(1)
* `remove_min`: O(N) (or O(1) if implemenbted as a linked list OR removing at the end) 

## Heaps

Instead of a list, we can store the items (with their priorities) in a **binary heap**. The heap guarantess O(log n) insertion time and O(log n) time for find_min / remove_min. It stores the items in a *complete* binary tree, stored in an array. 

**Recall:** In a **complete binary tree** with height h, levels 0, 1, 2, ..., h-1 have the maximum number of nodes. On level h, all nodes are in the leftmost possible position at that level. 

A complete binary tree can be stored in an array (with the root at index 1, and index 0 left empty). 
* For a node at position $i$ (other than the root), its parent is in position $i//2$.
* For a node at position $i$, its left child is in position $2i$ and its right child in position $2i+1$.


A **binary heap** is a complete binary tree in which the keys stored in the nodes satisfy the **heap order property**:
  * for every node $p$ other than the root, the key stored at $p$ is greater than or equal to the key stored at $p$'s parents. 

<img src="https://github.com/cucs-python/public/blob/main/w2132/lectures/figures/binary_heap1.png?raw=true" width="600px">

### Inserting into a heap

* Insert the key into the last available slot in the heap array. This is the rightmost leaf on the last level.
* **percolate up/Up-heap bubbling**: Compare the new key to its parent, if the parent is greater, swap the keys. Repeat swapping with the parent until the heap-order property is restored, i.e. the parent is smaller than the new key.

<img src="https://github.com/cucs-python/public/blob/main/w2132/lectures/figures/binary_heap2.png?raw=true" width="600px">
<img src="https://github.com/cucs-python/public/blob/main/w2132/lectures/figures/binary_heap3.png?raw=true" width="600px">
<img src="https://github.com/cucs-python/public/blob/main/w2132/lectures/figures/binary_heap4.png?raw=true" width="600px">

In [18]:
class BinaryHeap:
    
    def __init__(self):
        self._data = [None] 

    def __len__(self): 
        return len(self._data)-1 # to account for empty index 0
    
    def insert(self, k):

        self._data.append(k) # insert key in last available slot

        current_idx = len(self._data)-1
        
        while current_idx > 1 and self._data[current_idx // 2] > k: #parent still greater than new key
            self._data[current_idx] = self._data[current_idx // 2]
            current_idx = current_idx // 2

        self._data[current_idx] = k
                    

    def __repr__(self): 
        return str(self._data)

In [19]:
heap = BinaryHeap()
heap.insert(4)
heap

[None, 4]

In [20]:
heap.insert(3)
heap.insert(2)
heap.insert(1)

In [21]:
heap

[None, 1, 2, 3, 4]

In [22]:
heap.insert(2)
heap

[None, 1, 2, 3, 4, 2]

In [24]:
heap.insert(0)

In [25]:
heap

[None, 0, 2, 1, 4, 2, 3]

### Removing the Minimum
The smallest key in the heap is always at the root (i.e. at index 1 in the array). 

* Store the root so we can eventually return it. 
* Overwrite the root with the rightmost leaf (i.e. the last key in the array). 
* **Percolate down**/**Down-heap bubbling** Select the *smaller* child of the root. If the key at root is greater than the key at the smaller child, swap it with the root. Repeat swapping the parent with the smaller child until the heap order property is restored.

<img src="https://github.com/cucs-python/public/blob/main/w2132/lectures/figures/binary_heap5.png?raw=true" width="600px">
<img src="https://github.com/cucs-python/public/blob/main/w2132/lectures/figures/binary_heap6.png?raw=true" width="600px">
<img src="https://github.com/cucs-python/public/blob/main/w2132/lectures/figures/binary_heap7.png?raw=true" width="600px">
<img src="https://github.com/cucs-python/public/blob/main/w2132/lectures/figures/binary_heap8.png?raw=true" width="600px">



In [29]:
class BinaryHeap:
    
    def __init__(self):
        self._data = [None] 

    def __len__(self): 
        return len(self._data)-1 # to account for empty index 0
    
    def insert(self, k):
        self._data.append(k) # insert key in last available slot

        current_idx = len(self._data)-1
        
        while current_idx > 1 and self._data[current_idx // 2] > k: #parent still greater than new key
            self._data[current_idx] = self._data[current_idx // 2]
            current_idx = current_idx // 2

        self._data[current_idx] = k
                    
    def __repr__(self): 
        return str(self._data)

    def find_min(self):  # runs in constant time O(1)
        return self._data[1]
    
    def remove_min(self):

        result = self._data[1]
                
        key = self._data[-1]
        self._data[1] = key
        self._data.pop() # remove last leaf 

        if len(self._data) > 1:
            current_idx = 1
            while current_idx * 2 < len(self._data): #as long as there is a left child 
                
                child_idx = current_idx * 2
                # first part of the condition checks if right child exists
                # second part checks if right child is less than left child 
                if child_idx + 1 < len(self._data) and self._data[child_idx] > self._data[child_idx+1]:
                    child_idx += 1
        
                if self._data[current_idx] > self._data[child_idx]:
                    self._data[current_idx] = self._data[child_idx]
                    current_idx = child_idx 
                else: 
                    self._data[current_idx] = key
                    return result 
    
            self._data[current_idx] = key
            
        return result #reached leaf node

In [30]:
heap = BinaryHeap()
heap.insert(4)
heap.insert(3)
heap.insert(2)
heap.insert(1)
heap

[None, 1, 2, 3, 4]

In [31]:
heap.find_min()

1

In [32]:
heap.remove_min()

1

In [34]:
heap

[None, 2, 4, 3]

In [35]:
heap.remove_min()

2

In [36]:
heap

[None, 3, 4]

In [37]:
heap.remove_min()

3

In [38]:
heap

[None, 4]

In [39]:
heap.remove_min()

4

In [40]:
heap

[None]

### Heap Sort

As mentioned earlier in the course, general comparison-based sorting is $\Omega (N \log N)$. We can prove that a faster sorting algorithm is not possible (although we will not do this here). 

We can use a binary heap to implement a $O(N \log N)$ sorting algorithm.
Recall that the height of a *complete* binary tree is $log_2(N+1)-1 = O(\log N)$. 
So each percolate up or percolate down pass takes $O(\log N)$.


* First, Insert each of the $N$ item into the heap. Each insertion costs $O(\log N)$ for a total of $O(N \log N)$.  
* Then, repeatedly use remove_min until the heap is empty and append each value to a new list. The total is again $O(N \log N)$
* The total running time is therefore $O(N \log N)$.


In [375]:
h = BinaryHeap()

li = [7,4,3,5,6,9,1]

for key in li: 
    h.insert(key)

result = []
while len(h) > 0: 
    result.append(h.remove_min())

In [377]:
result

[1, 3, 4, 5, 6, 7, 9]

### In-Place Heap Sort

The Heap Sort algorithm above requires $O(N)$ storage, because we need to write the values into a new array. In-place sorting means to re-arrange the keys without any additional storage (other than a constant amount of storage for temporary data during comparisons and copying). 

Here is a clever trick: After removing the smallest value from the heap, we append it at the end of the array (in the space we just freed up by removing). 

Even better, we can change the priority order of the heap so that for every node (other than the root) its parent has a *larger* value. The largest value will then be at the root of the tree. This is called a **max heap**. 

### The heapify operation

The *heapify* operation takes an unsorted array and converts it into a binary heap in-place. We can show that this operation runs in $O(n)$ time.

Algorithm:
* Start at the parent node of the last leaf, which is k = len(heap) // 2.
* For each i = k ... 1, percolate the value at $i$ down until it is in the correct position.

### Python's heapq module